# Training: LP and LoRA on D-bias / D-align (Section III-B)

Trains the DINOv2 ViT-L/14 cells C1–C4 of the paper and saves, for every run, the head (and LoRA adapter), validation posteriors and metrics. Evaluation on Synthbuster and Chameleon is a separate notebook that loads these runs.

| Cell | Strategy | Training version |
|---|---|---|
| C1 | LP | D-bias |
| C2 | LP | D-align |
| C3 | LoRA | D-bias |
| C4 | LoRA | D-align |

**CLIP replication (16 Oct):** the frozen DINOv2 protocol applied unchanged to OpenAI CLIP ViT-L/14, the frozen backbone of UnivFD [9] (its 224-px position embeddings interpolated to the 336-px input, as DINOv2's are from 518 px) (no separate pilot; LR selection repeated by the same rule). Start a fresh session: do NOT attach the DINOv2 training output. Freeze record: `protocol_frozen_clip.json`.

**Protocol after the pilot (session 1, 15 Oct):** LoRA 1,500 steps (pilot: validation AUROC 0.99997 at step 500, training loss < 1e-3 by step 1,500), LP 3,000 steps, gradient checkpointing off (6.1 GB peak, 35% faster), LP and LoRA grids unchanged. With these, `STAGES = ["lp", "lora_grid", "lora_seeds"]` runs all of C1–C4 in one session of about 8 h on T4 ×2.

**Originally run in stages**, one Kaggle session (Save Version → Save & Run All) per row. Set `STAGES` in section 1 before each session:

| Session | `STAGES` | What runs | Rough time on T4 ×2 |
|---|---|---|---|
| 1 | `["pilot"]` | one LoRA pilot run (pilot seed, not one of the 5); LP feature extraction + LP pilot | measure on your own GPU |
| — | | **check the pilot report (section 9), confirm or change the † values, then freeze** | |
| 2 | `["lp", "lora_grid"]` | all LP runs (3 LRs × 5 seeds × 2 versions, seconds each) and the 6 first-seed LoRA grid runs | ~7 h |
| 3 | `["lora_seeds"]` | the 8 remaining LoRA runs at the selected LR (seeds 1–4 × 2 versions) | ~9 h, may need a 4th session |

A session's output is only kept if the commit finishes, so jobs that would not finish before `SESSION_HOURS` are skipped and picked up next session. **To continue in the next session, add this notebook's previous version output as input** (Add Input → Your Work → this notebook); the `thesis_runs` folder is copied in and finished runs are skipped.

**Live progress:** set `NTFY_TOPIC` in section 1 and open `https://ntfy.sh/<topic>` before committing. You get a message when the session starts, when the data is mounted, every 10 minutes during training, and at the end. If "setup OK" does not arrive within ~20 minutes of the run starting, the dataset did not mount: stop the run so it does not use quota.

**Kaggle setup:** Accelerator **GPU T4 ×2**, Internet **On** (DINOv2 weights from Hugging Face), input = the build-dataset output (folder containing `d_bias.csv`).

**Freezing:** the first session that runs anything other than `pilot` writes `protocol_frozen.json` with the date, the protocol and a fingerprint of the protocol and the training code. Every later session refuses to run if either has changed.

**v2 safety changes:** feature/prediction batch default 8, LoRA checkpoints every 200 optimizer steps, worker-error propagation, and preflight. The resume checkpoint holds the LoRA weights, head and AdamW states (tens of MB) and is deleted when the run finishes. Checkpoint continuation requires copying the previous run output into the new session input. Do not change the protocol or trainer source after freezing. A notebook completion does **not** imply that deferred jobs have finished; check the result table.


## 1. Configuration

In [ ]:
import os, sys, json, time, shutil, hashlib, subprocess, datetime
from pathlib import Path

STAGES = ["lp", "lora_grid", "lora_seeds"]   # all training in one session (pilot done); reruns skip finished runs
SESSION_HOURS = 10.5        # Kaggle stops a session at 12 h; jobs that would end later are left for the next one
INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUNS = WORK / "thesis_runs"
BACKBONE = "clip"           # exploratory replication: the frozen DINOv2 protocol applied to UnivFD's backbone
FEATURE_BATCH = 8  # start with 8 on T4 16GB; raise only after measuring memory
PREDICT_BATCH = 8
CHECKPOINT_EVERY = 200  # optimizer steps; save for interrupted Kaggle sessions

# Everything that defines a run. Values marked † in Table II are confirmed in the pilot, then frozen.
PROTOCOL = {
    "weights": {"dinov2": "facebook/dinov2-large",        # official DINOv2 ViT-L/14
                "clip": "openai/clip-vit-large-patch14"},  # OpenAI CLIP ViT-L/14, the frozen backbone of UnivFD [9]
    "versions": ["d_bias", "d_align"],
    "input": "336x336 crop, no resizing; the backbone's own mean/std (ImageNet for DINOv2, CLIP's for CLIP); position embeddings interpolated to 336 px; random horizontal flip (p=0.5)",
    "head": "new nn.Linear on the final-layer class token (post-LayerNorm); sigmoid; BCE",
    "lora": {"r": 8, "alpha": 16, "dropout": 0.1,                                   # alpha †, dropout †
             "targets": "query, key, value and output projections of every block"},
    "optim": {"name": "AdamW", "weight_decay": 0.01,                                # †
              "schedule": "cosine to 0", "warmup_frac": 0.05,
              "micro_batch": 8, "accum": 4,                                         # batch 32 †
              "steps": 3000,                                                        # LP
              "steps_lora": 1500},   # † set from the pilot: val AUROC 0.99997 at step 500, train loss < 1e-3 by 1500
    "lr_grid": {"lp": [1e-4, 3e-4, 1e-3], "lora": [5e-5, 1e-4, 2e-4]},             # LoRA grid †
    "seeds": [0, 1, 2, 3, 4], "first_seed": 0,
    "selection": "per strategy, the LR with the highest mean validation AUROC over D-bias and D-align "
                 "(LP: also averaged over all seeds; LoRA: first seed only), AUROC rounded to 4 decimals; "
                 "ties broken by the lower mean validation BCE",
    "precision": "fp16 autocast with gradient scaling (LoRA); frozen-feature extraction in fp16",
    "grad_checkpointing": False,   # pilot probe: 6.1 GB peak, 35% faster; speed only, results unchanged
}
PILOT = {"version": "d_align", "lr": 1e-4, "seed": 1000, "eval_every": 500,
         "lp_lrs": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]}  # pilot only: does the LP optimum lie inside the grid?

# Live progress on your phone: Kaggle shows a commit's log only after it ends. Choose a hard-to-guess
# topic, open https://ntfy.sh/<topic> in a browser (or the ntfy app) before starting, and watch updates.
# Messages are public to anyone who knows the topic; they contain only progress lines. "" turns this off.
NTFY_TOPIC = ""

def notify(msg):
    """Best effort: never stops the run if the network or the service is unavailable."""
    if not NTFY_TOPIC:
        return
    try:
        import urllib.request
        urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{NTFY_TOPIC}",
                               data=str(msg)[:3500].encode(), method="POST"), timeout=10).read()
    except Exception as e:
        print("notify failed:", type(e).__name__)


T0 = time.time()
DEADLINE = T0 + SESSION_HOURS * 3600
notify(f"session started: STAGES={STAGES}")
RUNS.mkdir(parents=True, exist_ok=True)

## 2. Setup: packages, data, previous sessions, GPUs

In [ ]:
try:
    import peft
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
import numpy as np, pandas as pd, torch, transformers, peft

# PEFT refuses to add LoRA layers when torchao < 0.16 is installed (Kaggle ships 0.10). This project does not
# use torchao, so remove it; the GPU workers start as new processes and no longer see it.
import importlib, importlib.metadata as _md
from packaging.version import Version
try:
    _v = _md.version("torchao")
    if Version(_v) < Version("0.16.0"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
        importlib.invalidate_caches()
        print(f"removed incompatible torchao {_v} (PEFT needs >= 0.16 or none)")
except _md.PackageNotFoundError:
    pass

def find(root, name, want_dir=False, max_depth=8, max_seconds=600):
    """Paths named `name` under root. Follows symlinks but never enters the same real folder twice
    (a symlink pointing back to a parent would otherwise make the walk endless), limits the depth, and
    does not descend into image folders (thousands of files, never what we look for)."""
    out, seen, t0 = [], set(), time.time()
    root = str(root)
    for dp, dns, fns in os.walk(root, followlinks=True):
        real = os.path.realpath(dp)
        depth = dp[len(root):].count(os.sep)
        if real in seen or depth > max_depth:
            dns[:] = []
            continue
        seen.add(real)
        out += [Path(dp) / n for n in (dns if want_dir else fns) if n == name]
        dns[:] = [d for d in dns if d not in ("real", "fake_png", "fake_jpeg", "lora", "lp", "claims")]
        if time.time() - t0 > max_seconds:
            raise RuntimeError(f"searching {root} for {name} took over {max_seconds}s; stopped at {dp}")
    return sorted(out)

# Training data: the folder that holds d_bias.csv / d_align.csv
notify("setup: packages ready, searching inputs")
hits = find(INPUT_ROOT, "d_bias.csv")
assert len(hits) == 1, f"expected one d_bias.csv under {INPUT_ROOT}, found {hits}"
DATA_ROOT = hits[0].parent
db, da = (pd.read_csv(DATA_ROOT / f"{v}.csv") for v in PROTOCOL["versions"])
assert (db["id"].to_numpy() == da["id"].to_numpy()).all(), "the two versions must list the same ids in the same order"
assert (db["label"].to_numpy() == da["label"].to_numpy()).all()
print("data:", DATA_ROOT, "|", db.groupby(["split", "label"]).size().to_dict())
notify(f"setup: training data found at {DATA_ROOT}")

# Runs from earlier sessions (this notebook's previous output added as input)
for prev in [p for p in find(INPUT_ROOT, "thesis_runs", want_dir=True) if p.resolve() != RUNS.resolve()]:
    n = 0
    for src in (Path(dp) / f for dp, _, fs in os.walk(prev) for f in fs):
        dst = RUNS / src.relative_to(prev)
        if src.is_file() and not dst.exists() and "claims" not in src.parts:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            n += 1
    print(f"copied {n} files from {prev}")
    notify(f"setup: copied {n} files from previous output")

N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)] or "none")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert N_GPUS > 0 or BACKBONE.startswith("tiny"), "turn on a GPU accelerator (T4 x2)"

if BACKBONE in PROTOCOL["weights"]:  # download once here, so parallel workers do not race on the cache
    notify("setup: downloading DINOv2 weights")
    from huggingface_hub import snapshot_download
    # config + safetensors only: the repo also holds the same weights as pytorch_model.bin
    snapshot_download(PROTOCOL["weights"][BACKBONE], allow_patterns=["*.json", "*.safetensors"])
notify(f"setup OK: data mounted ({len(db)} rows), GPUs {N_GPUS}")

## 3. Training code
Written to a file so each GPU can run it in its own process. Its contents are part of the protocol fingerprint.

In [ ]:
%%writefile /kaggle/working/thesis_lib.py
"""Training code for Section III-B (Models and training).

Used two ways:
  * imported by the notebook (helpers, run names, selection), and
  * run as a GPU worker:  python thesis_lib.py QUEUE.json
    The worker runs the queued jobs one after another on the GPU in CUDA_VISIBLE_DEVICES.

Every job writes into RUNS/<kind>/<name>/ and finishes by writing metrics.json, so a finished job is
skipped when it is queued again (e.g. in the next Kaggle session).
"""
import json, math, os, sys, time, traceback, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve

CROP = 336
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"


# ----------------------------------------------------------------------------- naming and files
def run_name(backbone, strategy, version, lr, seed, tag=""):
    return f"{tag}{backbone}_{strategy}_{version}_lr{lr:g}_s{seed}"


def run_dir(runs, strategy, name):
    return Path(runs) / strategy / name


def is_done(runs, strategy, name):
    d = run_dir(runs, strategy, name)
    required = [d / "metrics.json", d / "head.pt", d / "val_preds.npz"]
    if strategy == "lora":
        required.append(d / "adapter" / "adapter_config.json")
    return all(p.is_file() and p.stat().st_size > 0 for p in required)


def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)


# ----------------------------------------------------------------------------- data
def load_split(data_root, version, split):
    """Rows of d_bias.csv / d_align.csv for one split, in file order (identical across versions)."""
    df = pd.read_csv(Path(data_root) / f"{version}.csv")
    return df[df["split"] == split].reset_index(drop=True)


def load_rgb(path):
    with Image.open(path) as im:
        a = np.asarray(im.convert("RGB"))
    if a.shape != (CROP, CROP, 3):
        raise ValueError(f"{path}: expected {CROP}x{CROP}, got {a.shape}")
    return a


def to_tensor(a):
    return torch.from_numpy(np.array(a, copy=True)).permute(2, 0, 1)  # uint8 CHW (copy: PIL arrays are read-only)


NORMS = {"dinov2": (IMAGENET_MEAN, IMAGENET_STD),   # each backbone's own pretraining normalization
         "clip": ((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))}
_NORM = NORMS["dinov2"]
_FWD_KW = {}   # extra forward arguments for the current backbone


def set_norm(backbone):
    """Normalization and forward arguments of the current backbone. CLIP ViT-L/14 is pretrained at 224 px, so its
    position embeddings are interpolated to the 336-px input (DINOv2 does this internally, from 518 px)."""
    global _NORM, _FWD_KW
    is_clip = "clip" in backbone
    _NORM = NORMS["clip"] if is_clip else NORMS["dinov2"]
    _FWD_KW = {"interpolate_pos_encoding": True} if is_clip else {}


def normalize(x):
    """uint8 NCHW on device -> float32 normalized with the current backbone's mean/std."""
    mean = torch.tensor(_NORM[0], device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(_NORM[1], device=x.device).view(1, 3, 1, 1)
    return (x.float() / 255.0 - mean) / std


def index_stream(n, total, seed):
    """Seeded order of training rows and horizontal flips: `total` draws as back-to-back permutations.

    Depends only on (n, total, seed), and the two training versions list the same identifiers in the
    same order, so one seed gives every cell (LP/LoRA x D-bias/D-align) the same data order and flips.
    """
    rng = np.random.default_rng([seed, 101])
    idx = np.concatenate([rng.permutation(n) for _ in range(math.ceil(total / n))])[:total]
    flip = np.random.default_rng([seed, 102]).random(total) < 0.5
    return idx, flip


class StreamSet(torch.utils.data.Dataset):
    def __init__(self, root, paths, labels, idx, flip):
        self.root, self.paths, self.labels, self.idx, self.flip = Path(root), paths, labels, idx, flip

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, k):
        i = self.idx[k]
        a = load_rgb(self.root / self.paths[i])
        if self.flip[k]:
            a = a[:, ::-1]
        return to_tensor(a), float(self.labels[i])


class PathSet(torch.utils.data.Dataset):
    def __init__(self, root, paths, flip=False):
        self.root, self.paths, self.flip = Path(root), paths, flip

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        a = load_rgb(self.root / self.paths[i])
        return to_tensor(a[:, ::-1] if self.flip else a)


def loader(ds, batch, workers):
    return torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=False, num_workers=workers,
                                       pin_memory=DEVICE == "cuda", persistent_workers=workers > 0,
                                       prefetch_factor=4 if workers > 0 else None)


# ----------------------------------------------------------------------------- models
def load_backbone(name, protocol):
    """Pretrained backbone returning the post-LayerNorm final class token as `pooler_output`."""
    from transformers import Dinov2Config, Dinov2Model
    if name == "dinov2":
        m = Dinov2Model.from_pretrained(protocol["weights"]["dinov2"], attn_implementation="sdpa")
    elif name == "tiny-test":  # random tiny DINOv2 for offline tests of the pipeline
        torch.manual_seed(12345)  # fixed random weights, like a pretrained checkpoint
        m = Dinov2Model(Dinov2Config(hidden_size=64, num_hidden_layers=2, num_attention_heads=4,
                                     intermediate_size=128, patch_size=14, image_size=518))
    elif name == "clip":   # OpenAI CLIP ViT-L/14 (UnivFD's backbone); pooler_output = post-LayerNorm class token
        from transformers import CLIPVisionModel
        m = CLIPVisionModel.from_pretrained(protocol["weights"]["clip"], attn_implementation="sdpa")
    elif name == "tiny-clip-test":
        from transformers import CLIPVisionConfig, CLIPVisionModel
        torch.manual_seed(12345)
        m = CLIPVisionModel(CLIPVisionConfig(hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                                             num_attention_heads=4, image_size=224, patch_size=14))
    elif name == "dda":
        raise NotImplementedError("DDA starting model: added after the DINOv2 cells are complete "
                                  "(load the released checkpoint and merge its adapter here).")
    else:
        raise ValueError(name)
    return m


def attention_linear_names(backbone):
    """Full names of the query, key, value and output projections of every block.

    Found by structure, because transformers v4 names them attention.attention.{query,key,value} and
    attention.output.dense, while v5 names them attention.{q,k,v,o}_proj.
    """
    names = [n for n, mod in backbone.named_modules()
             if isinstance(mod, nn.Linear) and (".attention." in f".{n}." or ".self_attn." in f".{n}.")
             and ".encoder.layer" in f".{n}"]   # DINOv2: encoder.layer.N.attention.*, CLIP: vision_model.encoder.layers.N.self_attn.*
    n_layers = backbone.config.num_hidden_layers
    if len(names) != 4 * n_layers:
        raise RuntimeError(f"expected 4 attention projections per block ({4 * n_layers}), found {len(names)}: "
                           f"{names[:8]}")
    return names


def add_lora(backbone, protocol):
    from peft import LoraConfig, get_peft_model
    lc = protocol["lora"]
    cfg = LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"], bias="none",
                     target_modules=attention_linear_names(backbone))
    return get_peft_model(backbone, cfg)


def make_head(dim, seed):
    """New linear head. Created right after seeding, so LP and LoRA share the head init for a seed."""
    torch.manual_seed(seed)
    return nn.Linear(dim, 1)


def lr_lambda(total, warmup_frac):
    warm = max(1, round(total * warmup_frac))

    def f(step):  # step = optimizer steps already taken
        if step < warm:
            return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
    return f


def make_optimizer(params, lr, protocol):
    o = protocol["optim"]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=o["weight_decay"])
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda(o["steps"], o["warmup_frac"]))
    return opt, sched


# ----------------------------------------------------------------------------- metrics
def val_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p, dtype=np.float64)
    pc = np.clip(p, 1e-7, 1 - 1e-7)
    tpr05 = float((p[y == 1] >= 0.5).mean())
    tnr05 = float((p[y == 0] < 0.5).mean())
    fpr, tpr, thr = roc_curve(y, p)
    bacc = (tpr + 1 - fpr) / 2
    k = int(np.argmax(bacc))
    return {"val_auroc": float(roc_auc_score(y, p)),
            "val_bce": float(-(y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean()),
            "val_bacc_05": (tpr05 + tnr05) / 2, "val_tpr_05": tpr05, "val_tnr_05": tnr05,
            # secondary threshold: maximizes balanced accuracy on this cell's own validation split
            "val_best_threshold": float(min(thr[k], 1.0)), "val_bacc_best": float(bacc[k])}


@torch.no_grad()
def predict(backbone, head, root, paths, batch, workers):
    backbone.eval()
    out = []
    for x in loader(PathSet(root, paths), batch, workers):
        x = normalize(x.to(DEVICE, non_blocking=True))
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            f = backbone(pixel_values=x, **_FWD_KW).pooler_output
        out.append(torch.sigmoid(head(f.float()).squeeze(1)).cpu())
    return torch.cat(out).numpy()


# ----------------------------------------------------------------------------- job: feature extraction (LP)
@torch.no_grad()
def job_extract(job):
    """Final-layer class token of every training-set file (both versions, both splits), unflipped and flipped."""
    P, data_root = job["protocol"], job["data_root"]
    out = Path(job["runs"]) / "features" / f"{job['backbone']}.npz"
    paths = sorted(set().union(*(set(pd.read_csv(Path(data_root) / f"{v}.csv")["path"]) for v in P["versions"])))
    bb = load_backbone(job["backbone"], P).to(DEVICE).eval()
    feats = []
    for flip in (False, True):
        chunks, t0 = [], time.time()
        for i, x in enumerate(loader(PathSet(data_root, paths, flip), job.get("batch", 8), job.get("workers", 3))):
            x = normalize(x.to(DEVICE, non_blocking=True))
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                chunks.append(bb(pixel_values=x, **_FWD_KW).pooler_output.float().cpu())
            if i % 50 == 0:
                log(f"extract flip={flip}: {sum(len(c) for c in chunks)}/{len(paths)} ({time.time() - t0:.0f}s)")
        feats.append(torch.cat(chunks).numpy())
    out.parent.mkdir(parents=True, exist_ok=True)
    np.savez(out, paths=np.array(paths), feats=np.stack(feats, axis=1).astype(np.float32))  # [n, 2, dim]
    log(f"features written: {out} {feats[0].shape}")


# ----------------------------------------------------------------------------- job: linear probe
def job_lp(job):
    """All LP runs of one backbone: versions x lrs x seeds, trained on cached class tokens."""
    P, runs, data_root = job["protocol"], job["runs"], job["data_root"]
    o = P["optim"]
    batch, steps = o["micro_batch"] * o["accum"], o["steps"]
    if not (Path(runs) / "features" / f"{job['backbone']}.npz").exists():
        job_extract(job)
    z = np.load(Path(runs) / "features" / f"{job['backbone']}.npz")
    row = {p: i for i, p in enumerate(z["paths"])}
    feats = torch.from_numpy(z["feats"])
    for version in job["versions"]:
        tr, va = load_split(data_root, version, "train"), load_split(data_root, version, "val")
        Ftr = feats[[row[p] for p in tr["path"]]].to(DEVICE)          # [n, 2, dim]
        Fva = feats[[row[p] for p in va["path"]], 0].to(DEVICE)       # unflipped
        ytr = torch.tensor(tr["label"].to_numpy(), dtype=torch.float32, device=DEVICE)
        for lr in job["lrs"]:
            for seed in job["seeds"]:
                name = run_name(job["backbone"], "lp", version, lr, seed, job.get("tag", ""))
                if is_done(runs, "lp", name):
                    continue
                t0 = time.time()
                head = make_head(Ftr.shape[-1], seed).to(DEVICE)
                opt, sched = make_optimizer(head.parameters(), lr, P)
                idx, flip = index_stream(len(tr), steps * batch, seed)
                idx_t, flip_t = torch.from_numpy(idx).to(DEVICE), torch.from_numpy(flip.astype(np.int64)).to(DEVICE)
                losses = []
                for s in range(steps):
                    sl = slice(s * batch, (s + 1) * batch)
                    # mean over 32 = the mean of 4 accumulated micro-batches of 8 used for LoRA
                    loss = F.binary_cross_entropy_with_logits(head(Ftr[idx_t[sl], flip_t[sl]]).squeeze(1), ytr[idx_t[sl]])
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    opt.step()
                    sched.step()
                    losses.append(loss.item())
                with torch.no_grad():
                    p = torch.sigmoid(head(Fva).squeeze(1)).cpu().numpy()
                d = run_dir(runs, "lp", name)
                d.mkdir(parents=True, exist_ok=True)
                torch.save(head.state_dict(), d / "head.pt")
                np.savez(d / "val_preds.npz", id=va["id"].to_numpy(), label=va["label"].to_numpy(), p=p)
                m = {"name": name, "backbone": job["backbone"], "strategy": "lp", "version": version, "lr": lr,
                     "seed": seed, "steps": steps, "batch": batch, "train_loss_last100": float(np.mean(losses[-100:])),
                     "loss_curve": [float(np.mean(losses[i:i + 50])) for i in range(0, steps, 50)],
                     "seconds": time.time() - t0, "protocol_fingerprint": job["fingerprint"], **val_metrics(va["label"], p)}
                (d / "metrics.json").write_text(json.dumps(m, indent=1))
                log(f"{name}: val AUROC {m['val_auroc']:.4f}, bAcc@0.5 {m['val_bacc_05']:.4f}")


# ----------------------------------------------------------------------------- job: LoRA
def atomic_torch_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def restore_rng(saved):
    torch.set_rng_state(saved["torch_cpu"])
    np.random.set_state(saved["numpy"])
    random.setstate(saved["python"])
    if DEVICE == "cuda" and saved.get("torch_cuda") is not None:
        torch.cuda.set_rng_state(saved["torch_cuda"])


def capture_rng():
    return {"torch_cpu": torch.get_rng_state(), "numpy": np.random.get_state(),
            "python": random.getstate(),
            "torch_cuda": torch.cuda.get_rng_state() if DEVICE == "cuda" else None}


def job_lora(job):
    P, runs, data_root = job["protocol"], job["runs"], job["data_root"]
    o = P["optim"]
    micro, accum, steps = o["micro_batch"], o["accum"], o.get("steps_lora", o["steps"])
    P = {**P, "optim": {**o, "steps": steps}}  # cosine schedule spans the LoRA steps
    if job.get("steps_override"):  # only for offline tests
        steps = job["steps_override"]
        P = {**P, "optim": {**o, "steps": steps}}
    name = job["name"]
    d = run_dir(runs, "lora", name)
    d.mkdir(parents=True, exist_ok=True)
    seed, lr, version = job["seed"], job["lr"], job["version"]
    ckpt = d / "resume.pt"
    ck_every = int(job.get("checkpoint_every", 200))
    pred_batch = int(job.get("predict_batch", 8))
    tr, va = load_split(data_root, version, "train"), load_split(data_root, version, "val")

    bb = load_backbone(job["backbone"], P)
    head = make_head(bb.config.hidden_size, seed)
    torch.manual_seed(seed)  # LoRA init and dropout
    bb = add_lora(bb, P)
    if P["grad_checkpointing"]:
        bb.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    bb.to(DEVICE).train()
    head.to(DEVICE).train()
    params = [p for p in bb.parameters() if p.requires_grad] + list(head.parameters())
    n_trainable = sum(p.numel() for p in params)
    opt, sched = make_optimizer(params, lr, P)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    idx, flip = index_stream(len(tr), steps * micro * accum, seed)
    first_step, curve, evals = 0, [], []
    if ckpt.exists():
        state = torch.load(ckpt, map_location="cpu", weights_only=False)
        if (state["fingerprint"], state["steps"], state["name"]) != (job["fingerprint"], steps, name):
            raise RuntimeError(f"Resume metadata mismatch for {ckpt}. Never silently resume a different protocol.")
        res = bb.load_state_dict(state["backbone"], strict=False)  # checkpoint holds the LoRA weights only
        if res.unexpected_keys or any("lora_" in k for k in res.missing_keys):
            raise RuntimeError(f"resume checkpoint does not match the LoRA layout of {name}")
        head.load_state_dict(state["head"])
        opt.load_state_dict(state["optimizer"])
        sched.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        first_step = int(state["next_step"])
        curve, evals = state["curve"], state["evals"]
        log(f"RESUMED {name} at step {first_step}/{steps}")
    # Start dataset at the correct draw; fixed per-step sample sequence and flip flags.
    offset = first_step * micro * accum
    ds = StreamSet(data_root, tr["path"].to_numpy(), tr["label"].to_numpy(), idx[offset:], flip[offset:])
    it = iter(loader(ds, micro, job.get("workers", 3)))
    if ckpt.exists(): restore_rng(state["rng"])  # DataLoader iterator construction consumes RNG
    eval_every = job.get("eval_every")  # pilot only: validation AUROC during training
    eval_ids = np.random.default_rng(0).permutation(len(va))[:job.get("eval_subset", 1000)]
    t0, t_log = time.time(), time.time()
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for s in range(first_step, steps):
        tot = 0.0
        for _ in range(accum):
            x, y = next(it)
            x, y = normalize(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, dtype=torch.float32)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                f = bb(pixel_values=x, **_FWD_KW).pooler_output
            loss = F.binary_cross_entropy_with_logits(head(f.float()).squeeze(1), y) / accum
            scaler.scale(loss).backward()
            tot += loss.item()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        sched.step()
        if not math.isfinite(tot):
            raise FloatingPointError(f"non-finite loss at step {s}")
        curve.append(tot)
        if (s + 1) % 50 == 0:
            el = time.time() - t0
            log(f"{name} step {s + 1}/{steps} loss {np.mean(curve[-50:]):.4f} lr {sched.get_last_lr()[0]:.2e} "
                f"{el / (s + 1):.2f}s/step ETA {el / (s + 1) * (steps - s - 1) / 60:.0f} min")
        if eval_every and (s + 1) % eval_every == 0 and s + 1 < steps:
            p = predict(bb, head, data_root, va["path"].to_numpy()[eval_ids], pred_batch, job.get("workers", 3))
            evals.append({"step": s + 1, "val_auroc_subset": float(roc_auc_score(va["label"].to_numpy()[eval_ids], p))})
            log(f"{name} step {s + 1}: val AUROC (subset of {len(eval_ids)}) {evals[-1]['val_auroc_subset']:.4f}")
            bb.train()
            head.train()
        if ck_every > 0 and ((s + 1) % ck_every == 0 or s + 1 == steps):
            # Capture RNG after all evaluation effects; resume is at the next optimizer step.
            atomic_torch_save({"name": name, "fingerprint": job["fingerprint"], "steps": steps,
                "next_step": s + 1, "head": head.state_dict(),
                # trainable weights only (~25 MB); the frozen DINOv2 weights are reloaded from the hub
                "backbone": {k: v for k, v in bb.state_dict().items() if "lora_" in k},
                "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                "scaler": scaler.state_dict(), "rng": capture_rng(), "curve": curve, "evals": evals}, ckpt)
            log(f"checkpoint {name}: {s+1}/{steps}")
    train_sec = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0

    p = predict(bb, head, data_root, va["path"].to_numpy(), pred_batch, job.get("workers", 3))
    bb.save_pretrained(d / "adapter")
    torch.save(head.state_dict(), d / "head.pt")
    np.savez(d / "val_preds.npz", id=va["id"].to_numpy(), label=va["label"].to_numpy(), p=p)
    m = {"name": name, "backbone": job["backbone"], "strategy": "lora", "version": version, "lr": lr, "seed": seed,
         "steps": steps, "batch": micro * accum, "trainable_params": n_trainable,
         "train_loss_last100": float(np.mean(curve[-100:])),
         "loss_curve": [float(np.mean(curve[i:i + 50])) for i in range(0, steps, 50)], "evals": evals,
         "seconds": time.time() - t0, "train_seconds": train_sec, "sec_per_step": train_sec / max(1, steps-first_step),
         "peak_mem_gb": peak, "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
         "protocol_fingerprint": job["fingerprint"], **val_metrics(va["label"], p)}
    (d / "metrics.json").write_text(json.dumps(m, indent=1))
    if ckpt.exists(): ckpt.unlink()
    log(f"{name} done in {m['seconds'] / 60:.0f} min: val AUROC {m['val_auroc']:.4f}, "
        f"bAcc@0.5 {m['val_bacc_05']:.4f}, peak {peak:.1f} GB")


# ----------------------------------------------------------------------------- worker
def job_probe(job):
    """Pilot only: a short LoRA run WITHOUT gradient checkpointing, to measure peak memory and speed.
    Checkpointing only trades compute for memory and does not change the result."""
    P = {**job["protocol"], "grad_checkpointing": False}
    job_lora({**job, "protocol": P, "steps_override": job.get("probe_steps", 30), "checkpoint_every": 0})


JOBS = {"extract": job_extract, "lp": job_lp, "lora": job_lora, "probe": job_probe}


def job_done(job):
    if job["kind"] == "extract":
        return (Path(job["runs"]) / "features" / f"{job['backbone']}.npz").exists()
    if job["kind"] in ("lora", "probe"):
        return is_done(job["runs"], "lora", job["name"])
    return False  # lp skips finished runs itself


def claim(claims_dir, label):
    """Atomically claim a job, so that each job of the shared list runs on exactly one GPU."""
    Path(claims_dir).mkdir(parents=True, exist_ok=True)
    try:
        os.close(os.open(Path(claims_dir) / label, os.O_CREAT | os.O_EXCL))
        return True
    except FileExistsError:
        return False


def worker(queue_path):
    failed = 0
    q = json.loads(Path(queue_path).read_text())
    for job in q["jobs"]:
        label = job.get("name", job["kind"])
        if job_done(job):
            log(f"skip (already done): {label}")
            continue
        if not claim(q["claims"], label):
            continue  # taken by the other GPU
        est = q["est_sec"].get(job["kind"], 0)
        if time.time() + est > q["deadline"]:
            log(f"skip (would pass the session deadline, needs ~{est / 3600:.1f} h): {label}")
            continue
        log(f"start: {label}")
        set_norm(job.get("backbone", "dinov2"))
        try:
            JOBS[job["kind"]](job)
        except Exception:
            failed += 1
            err = traceback.format_exc()
            log(f"FAILED: {label}\n{err}")
            d = Path(job["runs"]) / "errors"
            d.mkdir(parents=True, exist_ok=True)
            (d / f"{label}.txt").write_text(err)
        finally:
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
    log(f"worker finished; failures={failed}")
    if failed: raise RuntimeError(f"{failed} job(s) failed; inspect errors/ and worker logs")


if __name__ == "__main__":
    worker(sys.argv[1])

## 4. Protocol fingerprint and freeze

In [ ]:
LIB_PATH = WORK / "thesis_lib.py"
sys.path.insert(0, str(WORK))
import importlib, thesis_lib
importlib.reload(thesis_lib)
from thesis_lib import run_name, run_dir, is_done

lib_src = LIB_PATH.read_text()
FINGERPRINT = hashlib.sha256((json.dumps(PROTOCOL, sort_keys=True) + lib_src).encode()).hexdigest()[:16]
print("protocol fingerprint:", FINGERPRINT)

FROZEN = RUNS / f"protocol_frozen_{BACKBONE}.json"   # one freeze record per backbone
if FROZEN.exists():
    fz = json.loads(FROZEN.read_text())
    if fz["fingerprint"] != FINGERPRINT:
        raise RuntimeError(f"The protocol or training code changed since it was frozen on {fz['frozen_utc']} "
                           f"(fingerprint {fz['fingerprint']} -> {FINGERPRINT}). Undo the change, or record it as "
                           "a deviation and delete protocol_frozen.json deliberately.")
    print(f"protocol frozen on {fz['frozen_utc']}: unchanged")
elif set(STAGES) - {"pilot"}:
    FROZEN.write_text(json.dumps({
        "frozen_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "fingerprint": FINGERPRINT, "lib_sha256": hashlib.sha256(lib_src.encode()).hexdigest(),
        "protocol": PROTOCOL, "torch": torch.__version__, "transformers": transformers.__version__,
        "peft": peft.__version__}, indent=1))
    print("PROTOCOL FROZEN now. Record this date, the fingerprint and this notebook's Kaggle version (or GitHub commit).")
else:
    print("pilot stage: protocol not frozen yet")

## 5. Job runner
One worker process per GPU. Workers take jobs from one shared list (whichever GPU is free takes the next job), skip finished jobs, and do not start a job that would end after the session deadline.

In [ ]:
def lora_job(version, lr, seed, tag="", **extra):
    return {"kind": "lora", "name": run_name(BACKBONE, "lora", version, lr, seed, tag), "backbone": BACKBONE,
            "version": version, "lr": lr, "seed": seed, **extra}


def lp_job(seeds, lrs, tag=""):
    return {"kind": "lp", "name": f"{tag}{BACKBONE}_lp_all", "backbone": BACKBONE,
            "versions": PROTOCOL["versions"], "seeds": seeds, "lrs": lrs, "tag": tag}


def metrics(strategy, tag=""):
    rows = [json.loads(p.read_text()) for p in (RUNS / strategy).glob("*/metrics.json")]
    rows = [r for r in rows if r["name"].startswith(tag + BACKBONE + "_")]
    return pd.DataFrame(rows)


def est_seconds():
    lr = metrics("lora")
    lora = 1.1 * lr["seconds"].median() if len(lr) else 3.5 * 3600
    if not len(lr) and len(metrics("lora", "pilot_")):
        lora = 1.1 * metrics("lora", "pilot_")["seconds"].median()
    feats = (RUNS / "features" / f"{BACKBONE}.npz").exists()
    return {"lora": float(lora), "lp": 600.0 if feats else 3600.0, "extract": 3600.0}


def run_jobs(jobs, poll=60, show_every=600):
    common = {"runs": str(RUNS), "data_root": str(DATA_ROOT), "protocol": PROTOCOL, "fingerprint": FINGERPRINT,
              "workers": 2 if N_GPUS > 1 else 3, "batch": FEATURE_BATCH,
              "predict_batch": PREDICT_BATCH, "checkpoint_every": CHECKPOINT_EVERY}
    jobs = [{**common, **j} for j in jobs]
    session = time.strftime("%Y%m%d-%H%M%S")
    qf = RUNS / "logs" / f"queue_{session}.json"
    qf.parent.mkdir(parents=True, exist_ok=True)
    qf.write_text(json.dumps({"jobs": jobs, "deadline": DEADLINE, "est_sec": est_seconds(),
                              "claims": str(RUNS / "claims" / session)}))
    print(f"{len(jobs)} jobs queued; estimates (h): { {k: round(v / 3600, 2) for k, v in est_seconds().items()} }")
    procs = []
    for g in range(max(1, N_GPUS)):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(g), "PYTHONUNBUFFERED": "1"}
        logf = RUNS / "logs" / f"{session}_gpu{g}.log"
        procs.append((g, logf, subprocess.Popen([sys.executable, str(LIB_PATH), str(qf)], env=env,
                                                stdout=open(logf, "w"), stderr=subprocess.STDOUT, cwd=str(WORK))))
    last = 0
    while any(p.poll() is None for *_, p in procs):
        time.sleep(poll)
        if time.time() - last > show_every:
            last = time.time()
            lines = []
            for g, logf, p in procs:
                tail = logf.read_text().strip().splitlines()[-1:] if logf.exists() else []
                lines.append(f"[gpu{g}{' done' if p.poll() is not None else ''}] {tail[0] if tail else ''}")
            print("\n".join(lines), flush=True)
            notify(f"{(time.time() - T0) / 3600:.1f} h\n" + "\n".join(lines))
    for g, logf, p in procs:
        print(f"--- gpu{g} (exit {p.returncode}), last lines:\n" + "\n".join(logf.read_text().splitlines()[-5:]))
    # Report failures without raising: an error would stop the remaining cells (selection, report) and could
    # cost the session's finished runs. Failed jobs are re-queued automatically in the next session.
    errs = sorted((RUNS / "errors").glob("*.txt")) if (RUNS / "errors").exists() else []
    errs = [e for e in errs if e.stat().st_mtime > T0]  # this session only (copied files keep their old mtime)
    if errs:
        print(f"WARNING: {len(errs)} failed job(s): {[e.stem for e in errs]}\n" + errs[-1].read_text()[-3000:])
        notify(f"FAILED jobs: {[e.stem for e in errs]}\n{errs[-1].read_text().strip().splitlines()[-1][:300]}")
    bad = [g for g, logf, p in procs if p.returncode != 0]
    if bad:
        print(f"WARNING: workers {bad} exited with an error; inspect logs in {RUNS / 'logs'}")


def select_lr(strategy):
    """Section III-B selection rule (see PROTOCOL['selection'])."""
    df = metrics(strategy)
    lrs = PROTOCOL["lr_grid"][strategy]
    seeds = PROTOCOL["seeds"] if strategy == "lp" else [PROTOCOL["first_seed"]]
    df = df[df["lr"].isin(lrs) & df["seed"].isin(seeds)]
    need = len(lrs) * len(seeds) * len(PROTOCOL["versions"])
    if len(df) != need:
        raise RuntimeError(f"{strategy}: {len(df)} of {need} runs needed for LR selection are finished")
    t = df.groupby("lr").agg(mean_val_auroc=("val_auroc", "mean"), mean_val_bce=("val_bce", "mean")).reset_index()
    for v in PROTOCOL["versions"]:
        t[f"auroc_{v}"] = t["lr"].map(df[df.version == v].groupby("lr")["val_auroc"].mean())
    t["rank_key"] = t["mean_val_auroc"].round(4)
    t = t.sort_values(["rank_key", "mean_val_bce"], ascending=[False, True]).reset_index(drop=True)
    chosen = float(t.loc[0, "lr"])
    out = {"strategy": strategy, "selected_lr": chosen, "rule": PROTOCOL["selection"],
           "tie_broken_by_bce": bool((t["rank_key"] == t.loc[0, "rank_key"]).sum() > 1),
           "table": t.drop(columns="rank_key").to_dict("records"), "fingerprint": FINGERPRINT}
    (RUNS / f"selection_{BACKBONE}_{strategy}.json").write_text(json.dumps(out, indent=1))
    print(t.drop(columns="rank_key").to_string(index=False))
    print(f"{strategy}: selected lr = {chosen:g}" + ("  (tie on AUROC, broken by BCE)" if out["tie_broken_by_bce"] else ""))
    return chosen

## Preflight before starting GPU training
Checks dataset references and target layers. The following pilot remains required to measure memory and runtime.

In [ ]:
# PRE-FLIGHT: check a few image paths, checkpoint sizes and LoRA layer targeting before expensive runs.
for v in PROTOCOL["versions"]:
    csv = pd.read_csv(DATA_ROOT / f"{v}.csv")
    assert set(csv["split"]) == {"train", "val"}
    for rel in csv["path"].sample(min(8, len(csv)), random_state=0):
        assert (DATA_ROOT / rel).is_file(), f"missing image: {DATA_ROOT / rel}"
print("Dataset paths: OK")
thesis_lib.set_norm(BACKBONE)
if True:
    import torch.nn as nn
    temp_bb = thesis_lib.load_backbone(BACKBONE, PROTOCOL)
    matched = thesis_lib.attention_linear_names(temp_bb)
    print(f"LoRA target validation: {len(matched)} attention projections; expected {4 * temp_bb.config.num_hidden_layers}")
    assert len(matched) == 4 * temp_bb.config.num_hidden_layers
    # Add the LoRA layers and run one forward/backward pass on one image (CPU, so the GPUs stay free).
    # Catches library problems (e.g. PEFT/torchao) in about a minute instead of inside the GPU workers.
    temp_bb = thesis_lib.add_lora(temp_bb, PROTOCOL)
    temp_bb.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    temp_bb.train()
    x = thesis_lib.normalize(thesis_lib.to_tensor(thesis_lib.load_rgb(DATA_ROOT / db["path"].iloc[0]))[None])
    temp_bb(pixel_values=x, **thesis_lib._FWD_KW).pooler_output.sum().backward()
    n_grad = sum(p.grad is not None for n, p in temp_bb.named_parameters() if "lora_" in n)
    assert n_grad == 2 * len(matched), f"LoRA gradients missing: {n_grad} of {2 * len(matched)}"
    print(f"LoRA forward/backward on CPU: OK ({n_grad} LoRA tensors received gradients)")
    del temp_bb, x
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Preflight passed. This does NOT test 336px forward/backward memory; run pilot.")

## 6. Stage `pilot`
A 30-step probe without gradient checkpointing (memory and speed only, about 2 minutes), then one full-length LoRA run at the default values on a pilot seed (1000, not one of the five), with validation AUROC every 500 steps; on the other GPU, LP features are extracted (kept for later) and LP is trained at every grid LR on the pilot seed. Pilot runs are named `pilot_*` and never enter the analysis.

In [ ]:
if "pilot" in STAGES:
    run_jobs([{**lora_job(PILOT["version"], PILOT["lr"], PILOT["seed"], tag="probe_"), "kind": "probe"},
              lora_job(PILOT["version"], PILOT["lr"], PILOT["seed"], tag="pilot_", eval_every=PILOT["eval_every"]),
              lp_job([PILOT["seed"]], PILOT["lp_lrs"], tag="pilot_")])

## 7. Stages `lp` and `lora_grid`
All LP runs, and the first-seed LoRA runs at every grid LR. LP is cheap, so it finishes long before the LoRA runs and its selection is made right away.

In [ ]:
jobs = []
if "lp" in STAGES:
    jobs.append(lp_job(PROTOCOL["seeds"], PROTOCOL["lr_grid"]["lp"]))
if "lora_grid" in STAGES:
    jobs += [lora_job(v, lr, PROTOCOL["first_seed"]) for lr in PROTOCOL["lr_grid"]["lora"] for v in PROTOCOL["versions"]]
if jobs:
    run_jobs(jobs)
if "lp" in STAGES:
    try:
        LP_LR = select_lr("lp")
    except RuntimeError as e:
        print("LP selection deferred to a continuation session:", e)

## 8. Stage `lora_seeds`
Finishes any first-seed grid runs the previous session left over, selects the LoRA LR from the first-seed grid, then trains the remaining seeds at that value. The first-seed run at the selected value is reused.

In [ ]:
if "lora_seeds" in STAGES:
    grid = [lora_job(v, lr, PROTOCOL["first_seed"]) for lr in PROTOCOL["lr_grid"]["lora"] for v in PROTOCOL["versions"]]
    if not all(is_done(RUNS, "lora", j["name"]) for j in grid):  # grid runs left over from the previous session
        run_jobs(grid)
    if not all(is_done(RUNS, "lora", j["name"]) for j in grid):
        print("LoRA grid incomplete; run this same stage again with previous outputs attached.")
    else:
        LORA_LR = select_lr("lora")
        run_jobs([lora_job(v, LORA_LR, s) for s in PROTOCOL["seeds"] if s != PROTOCOL["first_seed"]
              for v in PROTOCOL["versions"]])

## 9. Report
All finished runs. In the pilot session, check: the loss falls and flattens before step 3000, validation AUROC has stopped rising by the last checks, no failed jobs, peak memory well under 16 GB, and the projected GPU time.

In [ ]:
import matplotlib.pyplot as plt

for strategy in ("lp", "lora"):
    df = pd.concat([metrics(strategy), metrics(strategy, "pilot_")], ignore_index=True)
    if len(df):
        cols = ["name", "val_auroc", "val_bacc_05", "val_tpr_05", "val_tnr_05", "val_bce", "train_loss_last100", "seconds"]
        cols += [c for c in ("sec_per_step", "peak_mem_gb") if c in df]
        print(f"\n{strategy.upper()} runs ({len(df)})")
        print(df.sort_values("name")[cols].round(4).to_string(index=False))

pl = metrics("lora", "pilot_")
if len(pl):
    r = pl.iloc[0]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    steps = np.arange(len(r["loss_curve"])) * 50 + 50
    ax[0].plot(steps, r["loss_curve"]); ax[0].set(title="pilot LoRA: training loss (mean per 50 steps)", xlabel="step", yscale="log")
    if r["evals"]:
        e = pd.DataFrame(r["evals"])
        ax[1].plot(e["step"], e["val_auroc_subset"], "o-")
    ax[1].axhline(r["val_auroc"], ls="--", c="gray", label="final (full val)")
    ax[1].set(title="pilot LoRA: validation AUROC", xlabel="step"); ax[1].legend()
    plt.tight_layout(); plt.show()
    n_lora = len(PROTOCOL["lr_grid"]["lora"]) * 2 + (len(PROTOCOL["seeds"]) - 1) * 2
    h = r["seconds"] * n_lora / 3600
    print(f"pilot LoRA: {r['sec_per_step']:.2f} s/step on {r['gpu']}, {r['seconds'] / 3600:.2f} h per run, "
          f"peak {r['peak_mem_gb']:.1f} GB")
    print(f"projected: {n_lora} DINOv2 LoRA runs = {h:.0f} GPU-hours, i.e. about {h / max(1, N_GPUS):.0f} session hours "
          f"on {max(1, N_GPUS)} GPU(s)")
pr = metrics("lora", "probe_")
if len(pr):
    q = pr.iloc[0]
    print(f"without gradient checkpointing: {q['sec_per_step']:.2f} s/step, peak {q['peak_mem_gb']:.1f} GB "
          "(30 steps incl. warm-up, so s/step is slightly pessimistic)")
    if len(pl):
        print(f"  -> {1 - q['sec_per_step'] / pl.iloc[0]['sec_per_step']:.0%} faster per step than with checkpointing; "
              "safe to set grad_checkpointing=False before freezing if peak memory is below about 14 GB")
elif (RUNS / "errors").exists() and any(e.name.startswith("probe_") for e in (RUNS / "errors").iterdir()):
    err = next(e for e in (RUNS / "errors").iterdir() if e.name.startswith("probe_")).read_text().strip().splitlines()[-1]
    oom = "out of memory" in err.lower()
    print(f"without gradient checkpointing: FAILED -> {err[:200]}")
    print("  -> out of memory: keep grad_checkpointing=True" if oom else "  -> not a memory problem: fix the error and rerun the pilot")
print(f"\nsession time used: {(time.time() - T0) / 3600:.2f} h")

## 10. Training status
Every run the design needs (pilot runs excluded), and whether it is finished. Training is complete when this cell prints **TRAINING COMPLETE**; until then it names the `STAGES` for the next session.

In [ ]:
V, SEEDS, S0 = PROTOCOL["versions"], PROTOCOL["seeds"], PROTOCOL["first_seed"]
groups = {  # name -> (strategy, run names)
    "LP: 3 LRs x 5 seeds x 2 versions": ("lp", [run_name(BACKBONE, "lp", v, lr, s)
                                               for v in V for lr in PROTOCOL["lr_grid"]["lp"] for s in SEEDS]),
    "LoRA grid: 3 LRs x first seed x 2 versions": ("lora", [run_name(BACKBONE, "lora", v, lr, S0)
                                                         for v in V for lr in PROTOCOL["lr_grid"]["lora"]]),
}
sel = {s: RUNS / f"selection_{BACKBONE}_{s}.json" for s in ("lp", "lora")}
if sel["lora"].exists():
    lora_lr = json.loads(sel["lora"].read_text())["selected_lr"]
    groups[f"LoRA seeds {[s for s in SEEDS if s != S0]} at the selected LR {lora_lr:g} x 2 versions"] = (
        "lora", [run_name(BACKBONE, "lora", v, lora_lr, s) for v in V for s in SEEDS if s != S0])

missing = {}
for title, (strategy, names) in groups.items():
    missing[title] = [n for n in names if not is_done(RUNS, strategy, n)]
    print(f"{len(names) - len(missing[title]):3d}/{len(names)}  {title}")
    for n in missing[title]:
        print("        missing:", n)
for s, f in sel.items():
    print(f"  {'yes' if f.exists() else 'no ':3s}  {s.upper()} learning-rate selection ({f.name})")
print(f"  {'yes' if FROZEN.exists() else 'no ':3s}  protocol frozen ({FROZEN.name})")

lp_title, grid_title = list(groups)[:2]
if missing[lp_title] or missing[grid_title] or not sel["lp"].exists():
    print('\nNOT FINISHED. Next session: STAGES = ["lp", "lora_grid"]'); notify('session done. Next: STAGES = ["lp", "lora_grid"]')
elif not sel["lora"].exists() or any(missing.values()):
    print('\nNOT FINISHED. Next session: STAGES = ["lora_seeds"]'); notify('session done. Next: STAGES = ["lora_seeds"]')
else:
    print(f"\nTRAINING COMPLETE: {sum(len(n) for _, n in groups.values())} runs "
          f"(LP lr {json.loads(sel['lp'].read_text())['selected_lr']:g}, LoRA lr {lora_lr:g}). "
          "Next: the evaluation notebook, with this version's output as input.")